In [1]:
import sys
import os

# Adiciona o diretório pai (raiz do projeto) ao path
sys.path.append(os.path.abspath(os.path.join('..')))

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Tokenizador

In [3]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

text = '''Theoretical analysis and computational modeling are important tools for characterizing what 
nervous systems do, determining how they function, and understanding why they operate in particular 
ways. Neuroscience encompasses approaches ranging from molecular and cellular studies to human psychophysics 
and psychology. Theoretical neuroscience encourages cross-talk among these sub-disciplines by constructing 
compact representations of what has been learned, building bridges between different levels of description, 
and identifying unifying concepts and principles. In this book, we present the basic methods used for these 
purposes and discuss examples in which theoretical approaches have yielded insight into nervous system function.'''

/home/canopus/miniconda3/envs/transformer/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
input = tokenizer(
    text,
    padding = 'max_length',
    truncation = True,
    max_lenght = 512,
    return_tensors = 'pt'
)

In [5]:
input_ids = input['input_ids']
print(type(input_ids))
print(input_ids.shape)

<class 'torch.Tensor'>
torch.Size([1, 512])


In [6]:
vocab_size = tokenizer.vocab_size
d_model = 512

In [7]:
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model)
input_embedding = embedding_layer(input_ids)

In [8]:
input_embedding.shape

torch.Size([1, 512, 512])

# Positional Enconding

In [9]:
class PosEmbeding(nn.Module):
    def __init__(self, seq, d_model):
        super().__init__()
        self.seq = seq
        self.d_model = d_model

        pe = self.sinusoid_function()
        self.register_buffer('pe', pe)

# Positional Embedding Function
    def sinusoid_function(self):

        pe = torch.zeros((self.seq, self.d_model))
        

        for pos in range(self.seq):
            for i in range(0, self.d_model, 2):
                pe[pos, i] = math.sin(pos / 10_000 ** (i/self.d_model))
                pe[pos, i+1] = math.cos(pos / 10_000 ** (i/self.d_model))

        return pe

# Mecanismo de Atenção

In [10]:
# Multi-Head-Attention Mecanism
class QKVAttention(nn.Module):
    def __init__(self, d_model, h, s, causal=False):
        super().__init__()

        self.h = h
        self.reduced_dimension = d_model // h
        self.seq = s
        self.d_model = d_model
        self.batch = 1 # Posso tornar isso dinâmico
        self.causal = causal

        if causal:
            self.register_buffer('mask', torch.tril(torch.ones(s, s)))
        else:
            self.mask = None

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):

        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)

        Q_i = Q.reshape(self.batch, self.seq, self.h, self.reduced_dimension).transpose(1, 2)
        K_i = K.reshape(self.batch, self.seq, self.h, self.reduced_dimension).transpose(1, 2)
        V_i = V.reshape(self.batch, self.seq, self.h, self.reduced_dimension).transpose(1, 2)

        score = (Q_i @ K_i.transpose(-2, -1)) / math.sqrt(self.reduced_dimension)

        if self.causal:        
            score = score.masked_fill(self.mask == 0, float('-inf'))
   
        weight = F.softmax(score, dim=-1) @ V_i
        weight = weight.transpose(1, 2).contiguous().view(self.batch, self.seq, self.d_model)


        return self.W_O(weight).squeeze(0)
    

class FeedForward(nn.Module):
    def __init__(self, seq, hidden_size, dropout:float=0.1):
        super().__init__()

        self.f1 = nn.Linear(seq, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.f2 = nn.Linear(hidden_size, seq)
        
    def forward(self, x):
        x = self.f1(x)
        x = F.relu(x) # Considerar usar outra função de ativação
        x = self.dropout(x)
        x = self.f2(x)
        return x

In [11]:
class TransformerBlock(nn.Module):
    def __init__(self, max_seq_len, d_model, n_heads):
        super().__init__()

        # Defining Masked Multi Head Attetion
        self.mmha = QKVAttention(
            d_model = d_model,
            h = n_heads,
            s = max_seq_len,
            causal = True 
        )

        # Defining FeedForward (applies a layer normalization)
        self.ff = FeedForward(
            seq = max_seq_len,
            hidden_size = 4 * d_model
        )

        self.add_norm = nn.LayerNorm(d_model)

    def forward(self, x):

        res = x 
        x = self.mmha(x) # Masked Multi Head Attention
        x = self.add_norm(x + res) # Layer Normalization
        
        res = x 
        x = self.ff(x) # Feed Forward
        x = self.add_norm(x + res) # Residual + Layer Norm

        return x



class Transformer(nn.Module):
    def __init__(self, max_seq_len, vocab_size, embeding_dim, n_layers, d_model, n_heads):
        super().__init__()

        self.embeding = nn.Embedding(vocab_size, embeding_dim)
        
        self.pos_embeding = PosEmbeding(
            seq = max_seq_len,
            d_model = embeding_dim
        )

        self.blocks = nn.ModuleList()
        for _ in range(n_layers):
            self.blocks.append(
                TransformerBlock(
                    d_model = d_model,
                    h = n_heads,
                    s = max_seq_len,
                    causal=True # True if Masked
                )
            )

    def forward(self, x):

        self.input_embedding = self.embeding(x)
        self.positional_embedding = (self.input_embedding + self.pos_embeding(self.input_embedding))

        return self.pos_embeding, self.input_embedding


# Testando Código

In [12]:
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model)
input_embedding = embedding_layer(input_ids)
positional_embedding = PosEmbeding(512,512).pe

x = input_embedding + positional_embedding

In [13]:
model = TransformerBlock(512, d_model, 2)
model

TransformerBlock(
  (mmha): QKVAttention(
    (W_Q): Linear(in_features=512, out_features=512, bias=False)
    (W_K): Linear(in_features=512, out_features=512, bias=False)
    (W_V): Linear(in_features=512, out_features=512, bias=False)
    (W_O): Linear(in_features=512, out_features=512, bias=False)
  )
  (ff): FeedForward(
    (f1): Linear(in_features=512, out_features=2048, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (f2): Linear(in_features=2048, out_features=512, bias=True)
  )
  (add_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
)

In [14]:
x_hat = model.forward(x)

In [15]:

probabilities = F.softmax(x_hat, dim=-1)

In [23]:
y = probabilities.sum(dim=2, keepdim=True)
y

tensor([[[1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.0000],
         [1.